# Notebook 30 — WACV evaluation artifact and audited-subset protocol

This notebook is an **implementation brief for Claude**. It is intentionally
unexecuted: Claude should preserve this cell, implement the stages in the
placeholder cells below, and put reusable logic in tested `pilot/*.py` modules.

The paper target is **WACV Evaluations & Datasets**. The contribution is an
evaluation failure mode, a stress test that exposes it, and a reproducible
human-audit subset. It is **not** a new model or a claim that stratified entropy
is a useful grading signal.

## Implementation contract for Claude

Read `CLAUDE.md` first, especially the binary-stratification retraction, frozen
result discipline, audit provenance, and licensing notes. Then inspect these
sources before editing:

- `pilot/tests/test_stratum_degeneracy.py`
- `pilot/plotting.py`: `compute_auroc`,
  `correctness_collapses_onto_prediction`, and `bias_only_null_auroc`
- `pilot/audit_diagnostics.py` and its tests
- `pilot/data.py` and `pilot/prompts.py`
- `pilot/29_confusion_examples_for_professor.ipynb`
- `results/scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv`
- the three human-audit sets under `reference/audit/`

Before changing anything, show `git status --short`. The worktree is already
dirty and those changes belong to the user. Do not overwrite or reformat
notebooks 23–29, existing results, snapshots, the paper, report, slides, or
scorer defaults. Use small reusable modules and tests; this notebook should
orchestrate them. No model inference, GPU, paid API, or LLM judging is allowed.

### Outcome required

Build one deterministic, CPU-only WACV artifact with three clearly separated
parts:

1. a general binary-stratification null/stress test;
2. a pinned, reconstructable manifest for the frozen FERMAT n=300 run and its
   existing 234-item human-audit union;
3. a blinded second-rater packet and agreement pipeline that remains pending
   until a real human fills it.

Never turn missing evidence into a result. If a source artifact, dataset
revision, license fact, or human label cannot be verified, fail with an
actionable message and record the limitation.

## A. Formal binary-stratification null

Implement tested reusable logic, preferably in
`pilot/stratification_null.py`. Do not change the behavior of the existing
`bias_only_null_auroc` helper.

For item i and repeat j, simulate a model with no item-level information:

- `V_ij ~ Bernoulli(p)` independently of the item and truth;
- use odd K and majority prediction
  `M_i = 1[sum_j V_ij > K/2]`;
- use binary Shannon entropy of the empirical vote fraction;
- for fixed binary truth y, define correctness `C_i = 1[M_i = y]`;
- compute AUROC with entropy predicting **error**, using the same orientation
  and tie handling as `pilot.plotting.compute_auroc`.

State and verify the identity explicitly:

- when y=1, `C=M`;
- when y=0, `C=1-M`;
- therefore, inside a fixed-label stratum, correctness is only a relabeling
  of the vote from which entropy was computed.

The figure and prose must say that any apparent signal here is voting
arithmetic, not item-level competence.

Run both fixed-truth strata y=0 and y=1 and a balanced pooled sanity condition.
Use `p = 0.05, 0.10, ..., 0.95` and odd
`K in {3, 5, 7, 9, 15}`. Declare `n_items=650` and at least 2,000 replications
for the final run, with independent deterministic seeds per grid cell. Use
chunking or multinomial/binomial sufficient statistics so the full grid is
fast and memory-safe on CPU. A smaller smoke mode is allowed, but it must be
visibly labeled and may not overwrite final artifacts.

Treat this p-by-K grid as a required reviewer-control experiment, not an
optional visualization. The frozen model result may remain at the original
evaluation setting K=5, but the null/stress test must show how the artifact
behaves for multiple sampling budgets. The notebook should answer the reviewer
question explicitly: K was not tuned after the fact; K=5 is the frozen protocol,
and the structural fixed-stratum artifact persists across the tested K values.

For every cell save: truth condition, p, K, n_items, n_sims, seed, valid and
invalid replicate counts, median AUROC, 2.5/97.5 percentiles, median minority
count and rate, and runtime. If a replicate contains only one correctness
class, report it as undefined; never replace undefined AUROC with 0.5. Add
checks for p=0.5 symmetry, y=0/y=1 complement behavior, and chance behavior in
the balanced pooled condition. If an exact binomial oracle is implemented,
test Monte Carlo against it but keep the simulation as the displayed stress
test.

Overlay or tabulate the three already locked K=5 cases, sourcing each value
from its CSV when available and otherwise from the tested frozen constants in
`test_stratum_degeneracy.py` with that fallback disclosed:

| model | n | fixed error-vote rate p | observed stratified AUROC |
|---|---:|---:|---:|
| Qwen2.5-VL-3B | 650 | 0.843 | 0.854 |
| Qwen2.5-VL-7B | 648 | 0.813 | 0.801 |
| LLaVA-NeXT-7B | 400 | 0.780 | 0.775 |

Also reproduce the perception counterexample from the frozen n=300 file:
38 unanimous items include 3 wrong transcriptions, while 4 maximum-entropy
items are correct. This is a structural contrast, not another headline result:
for open-ended transcription, agreement does not determine correctness.

Produce a tidy CSV, a machine-readable config/provenance JSON, and one
publication-ready heatmap or faceted curve with a 0.5 reference line and
undefined cells visibly distinguished. Use a new stable directory such as
`reference/wacv_evaluation_artifact/` and a paper-ready copy under
`paper/figures/` only if it does not overwrite an existing figure. Do not edit
`paper/main.tex` in this task.

## B. Reconstructable FERMAT n=300 audit artifact

Materialize the exact balanced 150/150, seed-42 sample behind
`results/scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv`.

FERMAT is gated and row order may change. Discover and record the authoritative
Hugging Face dataset revision/commit and original source-row indices. Fail
closed unless all 300 `orig_q`, `pert_a`, and `has_error` values align exactly
with the frozen CSV after reconstruction. Do not guess the revision and do not
accept partial or order-insensitive alignment. Record SHA-256 hashes for the
source CSV, prompts, relevant code/config, and per-item content. Preserve the
stored `transcription_correct` and entropy columns exactly; any recomputed
scorer must be a separately named diagnostic column.

Create two manifests:

- a **public metadata manifest** with stable item IDs 0–299, pinned
  dataset/revision/split/source-row identifiers, hashes, selection seed,
  `has_error`, model/run/prompt hashes, K, temperature, entropy, scorer version,
  and audit-selection provenance;
- a **private authorized manifest** that may include the gated text, page
  materialization references, stored raw generations, and extracted spans.

Do not put images, access tokens, private Drive paths, or gated dataset content
in the public artifact. Do not assume model generations derived from gated
pages are redistributable: include them publicly only if an explicit license
check supports that decision. If the dataset actually exposes `orig_a`, retain
it privately; never fabricate it when absent. Copy the exact prompts from
`pilot/prompts.py` and hash them rather than paraphrasing.

Preserve the first-rater audits in long form: one row per item per audit pass,
including source audit set/file, original label, note, confidence/coder fields
when present, and protocol provenance. Do not silently collapse overlaps.
The following invariants must remain explicit and tested:

- three targeted audit sets have 312 rows in total;
- their union has 234 unique items;
- 78 rows are overlaps;
- the four hard intra-rater contradictions remain visible and unadjudicated;
- targeted-set rates are not estimates for all 300 items.

Write a data card/README with schema, versioning, selection procedure,
license/gating and reconstruction instructions, intended use, known scorer
noise, single-rater limitation, and claims the artifact cannot support.

## C. Real second-rater protocol, with no fabricated labels

Create a deterministic queue over the 234-item audit union:

- a 120-item **agreement core**, sampled before any second-rater labels are
  seen and stratified across frozen scorer verdict, `has_error`, entropy bins,
  and first-audit source;
- a four-item **challenge set** containing the known contradictions, kept
  separate from the core and never mixed into representative agreement;
- an **extension queue** containing every remaining audited item, so the same
  protocol can cover the full union if rater time permits.

Record seed, stratum, selection probability, queue membership, and randomized
display order. The core estimates agreement only for the defined audited
union/design—not the whole FERMAT dataset or all 300 run items. If weighting is
used, document and test it.

Generate:

1. a blank blinded CSV/JSONL annotation template;
2. private HTML/contact sheets that load authorized FERMAT images locally;
3. a separate private key mapping randomized review IDs to item IDs;
4. concise rater instructions with examples and label definitions.

During coding, hide first-rater labels/notes, audit category, automatic
correct/wrong verdict, entropy, `has_error`, and whether the item is a challenge
case. Show only the page and neutral evidence needed to judge it: the frozen
model transcription/output and the dataset reference span or solution, labeled
without revealing the automatic decision. Require image inspection.

The human fields are:

- `model_correctness in {correct, wrong, indeterminate}`;
- `reference_fidelity in {faithful, unfaithful, indeterminate}`;
- `failure_category in {notation_misread, copied_wrong_line, hallucination,
  extraction_issue, reference_issue, ambiguous_multianswer, other,
  not_applicable}`;
- `confidence in {high, medium, low}`;
- required evidence/note;
- rater pseudonym, protocol version, and timestamp.

Define every label. `extraction_issue` and `indeterminate` must never be
automatically converted to model-wrong. Ship all second-rater label fields
blank. Never synthesize, infer, copy, or ask an LLM to supply a human label.

Until a genuinely completed file is provided, print exactly:

`SECOND-RATER STATUS: PENDING`

and refuse to calculate agreement. After a completed file is supplied, validate
IDs, allowed vocabulary, completeness, duplicates, blinding-key alignment, and
protocol version. Then report the preregistered outputs separately for the
random core and challenge set: confusion matrix, raw agreement, Cohen's kappa
with bootstrap interval where estimable, a clearly labeled determinate-only
sensitivity analysis, category agreement descriptively, and a disagreement
table for human adjudication. Do not adjudicate disagreements automatically.

## D. Required files and acceptance checks

Use stable, non-overwriting names under
`reference/wacv_evaluation_artifact/`, including at least:

- `null_grid.csv` and `null_grid_config.json`;
- `fermat_n300_public_manifest.csv`;
- `fermat_n300_private_manifest.csv` or `.jsonl`, excluded from public release;
- `audit_labels_long.csv`;
- `second_rater_queue.csv`;
- `second_rater_template_blank.csv`;
- `second_rater_instructions.md`;
- `README.md`;
- figures under a dedicated `figures/` directory;
- `agreement_summary.json` only after real second-rater completion.

Add focused tests, preferably
`pilot/tests/test_stratification_null.py` and
`pilot/tests/test_wacv_evaluation_artifact.py`, plus
`pilot/dryruns/dryrun_nb30.py`. Tests must cover:

- deterministic simulation and independent cell seeds;
- p=0.5 symmetry and fixed-stratum complement identity;
- Monte Carlo/oracle agreement if an oracle is added;
- all required K values `{3, 5, 7, 9, 15}` and all p values appear in the
  final grid for y=0, y=1, and balanced pooled conditions;
- reviewer-control reporting for the K sweep, including a K=5 marker for the
  frozen model protocol and a statement that K was not tuned;
- undefined one-class handling;
- source CSV hash and exactly 300 unique aligned rows, 150/150 by `has_error`;
- exactly five stored generations per arm where the frozen schema promises it;
- exact prompt hashes;
- no images, secrets, Drive paths, or gated text in public output;
- 312 long-form audit rows, union 234, overlaps 78, and all contradictions;
- core/challenge disjointness and complete queue coverage;
- every shipped second-rater answer blank;
- refusal to compute agreement from an incomplete or malformed template;
- fail-closed behavior without the pinned dataset revision.

The dry run must use synthetic/stub data and require no network or Hugging Face
login. Validate notebook JSON, run focused tests, then the full
`pytest pilot/tests -q` suite and `python3 paper/check_numbers.py` when
feasible. Clear notebook outputs before handoff.

Minimum commands to run and report after Stage A:

```bash
python3 -m pytest pilot/tests/test_stratification_null.py -q
python3 -m pytest pilot/tests/test_stratum_degeneracy.py -q
python3 -m pytest pilot/tests -q
python3 paper/check_numbers.py
python3 - <<'PY'
import json
from pathlib import Path
nb = json.loads(Path('pilot/30_wacv_evaluation_artifact.ipynb').read_text())
assert nb['nbformat'] == 4
assert all(cell.get('execution_count') is None for cell in nb['cells'] if cell['cell_type'] == 'code')
assert all(cell.get('outputs') == [] for cell in nb['cells'] if cell['cell_type'] == 'code')
print('notebook json ok')
PY
git diff --check
```

The final notebook cell must print:

- every created artifact path and SHA-256;
- source dataset revision and reconstruction status;
- test results and CPU runtime;
- second-rater status;
- what a WACV reviewer may claim;
- what remains forbidden to claim;
- the exact next human action.

### Interpretation contract

Allowed before second rating: the binary-stratification AUROC can arise under a
signal-free response bias; the public manifest makes the frozen evaluation
sample and audit selection verifiable subject to FERMAT access.

Not allowed before second rating: human-level accuracy, reliable inter-rater
agreement, a representative error rate for all FERMAT, a corrected benchmark
score, or any claim that the 234 targeted items are an IID evaluation set.

After implementing, summarize files changed, tests run, unresolved blockers,
and estimated human coding time. Do not add numerical conclusions to the paper
until the user reviews the artifacts and a real second rater completes the
protocol.


In [ ]:
# Stage 0 — environment, then inspect and freeze inputs.
# CPU-only: no GPU, no model, no API, no Hugging Face login, no Drive mount.
#
# This notebook needs `pilot` importable AND the repo's own data files on
# disk. Everything it reads -- the frozen run CSV and the audit CSVs -- is
# tracked in the repository, so a plain clone is sufficient and there is
# nothing to fetch from Drive. Notebooks 23-29 all begin with a cell like
# this; notebook 30 originally did not, which is why it failed with
# `ModuleNotFoundError: No module named 'pilot'` when run from a Jupyter or
# Colab kernel rather than from the repo root.
import os
import subprocess
import sys


def _at_repo_root() -> bool:
    return os.path.isdir("pilot") and os.path.isdir("reference/audit")


if not _at_repo_root():
    REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
    if not os.path.isdir("repo"):
        print("cloning the repository (public; no token needed)...")
        subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
    os.chdir("repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."],
                   check=False)

sys.path.insert(0, os.getcwd())
for _n in [m for m in list(sys.modules) if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_n]

# Import every module the later stages need, HERE, so a stale clone fails
# now with an actionable message instead of a bare ModuleNotFoundError
# halfway through Stage B. The clone comes from the REMOTE, so library code
# committed but not yet pushed is exactly the failure this catches.
_missing = []
for _m in ("pilot.stratification_null", "pilot.wacv_artifact",
           "pilot.second_rater"):
    try:
        __import__(_m)
    except ModuleNotFoundError:
        _missing.append(_m)
assert not _missing, (
    f"the clone predates this notebook's library code: {_missing}. Push those "
    "modules and re-run this cell -- a green local dry run does NOT cover it, "
    "because the dry run uses the working tree while this clones the remote.")
import pilot.stratification_null            # noqa: F401

assert _at_repo_root(), f"not at the repo root: {os.getcwd()}"
print(f"cwd            : {os.getcwd()}")
print(f"pilot imported : {os.path.dirname(pilot.stratification_null.__file__)}")

print("\n=== worktree BEFORE any edit (dirty files are the user's) ===")
print(subprocess.run(["git", "status", "--short"], capture_output=True,
                     text=True).stdout.rstrip() or "  (clean)")

import hashlib

RUN_CSV = "results/scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv"
AUDIT = ["reference/audit/coded_31_qwen_only_qwen_20260811.csv",
         "reference/audit/coded_73_wrong_on_both_20260811.csv",
         "reference/audit/spotcheck_40_qwen_strict_v1_correct_20260811.csv",
         "reference/audit/spotcheck_extra60_qwen_strict_v1_correct_20260812.csv",
         "reference/audit/strict_v2_high_priority_human_audit_20260812.csv"]
ARTIFACT_DIR = "reference/wacv_evaluation_artifact"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


print("\n=== frozen inputs, hashed BEFORE use ===")
missing = [f for f in [RUN_CSV] + AUDIT if not os.path.exists(f)]
for f in [RUN_CSV] + AUDIT:
    ok = os.path.exists(f)
    print(f"  {'OK ' if ok else 'MISSING'} {sha256(f)[:16] if ok else '-':>16}  {f}")
assert not missing, f"required inputs absent: {missing}"

# Assert CPU-only rather than promising it.
assert "torch" not in sys.modules, "no model inference in this notebook"
print("\nno torch imported; CPU-only artifact build")

In [ ]:
# Stage A — the binary-stratification null, as a REVIEWER CONTROL.
#
# WHY THE p-BY-K SWEEP EXISTS, and this framing is the point of the stage:
#
#   K=5 is the FROZEN EVALUATION PROTOCOL. It was fixed before any of these
#   results existed and is the K behind every reported number in this project.
#   It was NOT chosen after seeing which K made the null look worst.
#
#   The sweep over K in {3,5,7,9,15} and p in {0.05,...,0.95}, across y=0,
#   y=1 and a balanced pooled condition, is therefore a CONTROL, not a search.
#   It answers the reviewer question "is this artifact peculiar to your K, or
#   to your model's particular bias?" The answer is neither: the effect is
#   present at every K and grows smoothly with p, and it vanishes at p=0.5 and
#   under balanced pooling. A single K would leave that unanswered.
#
# Everything below is a SIGNAL-FREE responder: identical vote probability for
# every item, no item-level information whatsoever. Any AUROC it attains is
# voting arithmetic, and none of it is a measurement of any model.
import pandas as pd

import pilot.stratification_null as sn

print("identity, checked elementwise on real draws (not asserted in prose):")
print(" ", sn.collapse_identity(k=5, p=0.8, n_items=256, seed=3))

print("\nclosed form vs Monte Carlo, and the two strata as exact complements:")
for p in (0.5, 0.7, 0.843):
    a1 = sn.exact_stratum_auroc(p, 5, 1)["auroc"]
    a0 = sn.exact_stratum_auroc(p, 5, 0)["auroc"]
    print(f"  p={p:5.3f}  y=1 {a1:.4f}   y=0 {a0:.4f}   sum {a1 + a0:.12f}")

GRID_KW = dict(n_items=650, n_sims=2000, base_seed=20260814)
grid = sn.null_grid(**GRID_KW)          # 285 cells; ~3 min CPU
cfg = {
    "artifact": "binary-stratification null stress test",
    "role": "REVIEWER CONTROL, not a tuned result",
    "frozen_protocol_k": 5,
    "k_swept_because": "to show the artifact is not specific to the frozen K",
    "p_values": list(sn.DEFAULT_P), "k_values": list(sn.DEFAULT_K),
    "conditions": list(sn.TRUTH_CONDITIONS), **GRID_KW,
    "seed_scheme": "sha256(base|condition|p|k)[:4], independent per cell",
    "entropy_units": "nats",
    "entropy_fold": "folded on the INTEGER count so s and K-s tie exactly",
    "auroc": "pilot.plotting.compute_auroc, entropy predicts ERROR",
    "undefined_policy": "one-class replicates counted invalid, never 0.5",
    "smoke": False,
}
hashes = sn.write_grid(ARTIFACT_DIR, grid, cfg)
fig = sn.plot_null_grid(grid, f"{ARTIFACT_DIR}/figures/stratification_null.png")

d = grid[grid["exact_available"]]
pool = grid[grid["truth_condition"] == "pooled_balanced"]
print(f"\ncells {len(grid)} | MC vs exact max |diff| "
      f"{(d.auroc_median - d.auroc_exact).abs().max():.4f}")
print(f"balanced pooled median AUROC in [{pool.auroc_median.min():.4f}, "
      f"{pool.auroc_median.max():.4f}]  (chance, as required)")
print(f"one-class (undefined) replicates: {int(grid.n_invalid.sum())}, "
      "reported as invalid rather than substituted")

print("\nthe three LOCKED K=5 cases, against the null at their own bias rate:")
for name, n, p, obs in sn.LOCKED_CASES:
    null = sn.simulate_stratum(p, 5, 1, n_items=n, n_sims=2000, seed=20260814)
    print(f"  {name:<15} n={n:<4} p={p:.3f}  observed {obs:.3f}  "
          f"null median {null['auroc_median']:.3f}  "
          f"-> null {'EXCEEDS' if null['auroc_median'] > obs else 'below'} observed")
print("\nA responder with no item-level information outscores every model here,"
      "\nso the stratified figure evidences nothing about any model.")

In [ ]:
# Stage B — reconstructable manifest. OFFLINE ONLY; alignment is PENDING.
#
# FERMAT is gated, and metadata and data have DIFFERENT access levels:
# `dataset_info` succeeds anonymously and yields the commit sha, while
# `load_dataset` raises because the data is gated. So the revision can be
# pinned here; only the 300-row alignment needs an authenticated session.
# This stage never guesses a revision and never accepts a partial match.
import pandas as pd

import pilot.wacv_artifact as W

run = pd.read_csv(RUN_CSV)
assert len(run) == 300, f"expected 300 rows, got {len(run)}"

revision = W.dataset_revision()
print("pinned dataset revision:", revision["revision"], f"(gated={revision['gated']})")

alignment = W.verify_reconstruction(run)      # no `source` -> fails closed
print(f"alignment status : {alignment['alignment_status']}")
print(f"verified         : {alignment['verified']}")
print(f"reason           : {alignment['reason']}")
assert alignment["verified"] is False, (
    "offline reconstruction must NOT claim verification")

audit_long = W.audit_labels_long()
inv = W.audit_invariants(audit_long)
print("\naudit long form, overlaps preserved rather than merged:")
for k in ("n_rows", "n_unique_items", "n_overlap_rows", "hard_contradictions"):
    print(f"  {k:22s} {inv[k]}")
assert (inv["n_rows"], inv["n_unique_items"], inv["n_overlap_rows"]) == (312, 234, 78)
assert inv["hard_contradictions"] == [108, 149, 222, 230], (
    "the four intra-rater contradictions must stay visible and unadjudicated")

public, private = W.build_manifests(run, RUN_CSV, audit_long, revision, alignment)
print(f"\npublic manifest {public.shape}   private manifest {private.shape}")
print("safety check:", W.assert_public_manifest_is_safe(public, private))

provenance = {
    "dataset": revision, "split": W.DATASET_SPLIT, "selection": W.SELECTION,
    "protocol": W.RUN_PROTOCOL, "prompts": W.prompt_hashes(),
    "frozen_run": {k: v for k, v in W.frozen_run_hashes(RUN_CSV, run).items()
                   if k != "per_item_sha256"},
    "audit": inv, "alignment": {k: v for k, v in alignment.items()
                                if k != "source_row_index"},
    "claims_not_supported": [
        "human-level accuracy", "reliable inter-rater agreement",
        "a representative error rate for all of FERMAT",
        "a corrected benchmark score",
        "that the 234 audited items are an IID evaluation set",
    ],
}
written = W.write_artifact(ARTIFACT_DIR, public, private, audit_long, provenance)
for path, h in written.items():
    print(f"  {h[:16]}  {path}")
print("\nNEXT HUMAN ACTION for this stage: run it once in an authenticated"
      "\nColab session so `verify_reconstruction` can align all 300 rows"
      "\nagainst the pinned revision. Until then the manifest says PENDING.")

In [ ]:
# Stage C — blinded second-rater packet. TEMPLATE ONLY; no labels invented.
import pilot.second_rater as SR

queue = SR.build_queue(public, audit_long, run)
print(SR.queue_summary(queue))

template = SR.blank_template(queue)
assert SR.all_answers_blank(template), "shipped template must be empty"

paths = SR.write_packet(ARTIFACT_DIR, queue, template)
for path, h in paths.items():
    print(f"  {h[:16]}  {path}")

# The gate. No completed file exists, so no agreement may be computed.
status = SR.agreement_or_pending(ARTIFACT_DIR)
print()
print(status["message"])
assert status["state"] == "PENDING"

In [ ]:
# Claude — Stage D: validate artifacts, run tests, and generate paper-ready outputs.


In [ ]:
# Final handoff — paths, hashes, tests, status, and what may be claimed.
import subprocess

print("=== artifacts ===")
for root, _dirs, files in sorted(os.walk(ARTIFACT_DIR)):
    for f in sorted(files):
        fp = os.path.join(root, f)
        print(f"  {sha256(fp)[:16]}  {fp}")

print("\n=== tests ===")
for t in ("pilot/tests/test_stratification_null.py",
          "pilot/tests/test_wacv_evaluation_artifact.py"):
    r = subprocess.run([sys.executable, "-m", "pytest", t, "-q"],
                       capture_output=True, text=True)
    print(f"  {t}: {r.stdout.strip().splitlines()[-1] if r.stdout.strip() else 'no output'}")

print(f"\n=== status ===")
print(f"  dataset revision      : {revision['revision']}")
print(f"  reconstruction        : {alignment['alignment_status']}")
print(f"  SECOND-RATER STATUS: PENDING")

print("""
=== a WACV reviewer MAY claim from this artifact ===
  * the stratified AUROC can arise from a signal-free response bias, shown
    across K in {3,5,7,9,15} and p in 0.05..0.95, with the frozen K=5 among
    them rather than chosen after the fact;
  * the public manifest makes the frozen sample and the audit SELECTION
    verifiable, subject to FERMAT access.

=== NOT allowed before a real second rater ===
  * human-level accuracy;
  * reliable inter-rater agreement;
  * a representative error rate for all of FERMAT;
  * a corrected benchmark score;
  * that the 234 audited items are an IID evaluation set.

=== exact next human action ===
  1. run Stage B once in an AUTHENTICATED session to align the 300 rows;
  2. give the blinded packet to a second rater who has NOT seen the first
     audit, and return the completed CSV to re-run Stage C.
""")